# FRTB capital quickstart

This notebook is the first-run path for an upstream risk engine owner or integrator. It shows what tables your system must emit, which public package entrypoints to call, and how component results are composed into Standardised Approach capital.

All values and examples are synthetic engineering and validation evidence, not final regulatory capital, regulatory submissions, legal advice, or supervisory approval.

Start with the suite-level [client integration guide](../CLIENT_INTEGRATION.md), then use the package journeys for implementation detail: [SBM](../../packages/frtb-sbm/docs/PACKAGE_JOURNEY.md), [DRC](../../packages/frtb-drc/docs/PACKAGE_JOURNEY.md), [RRAO](../../packages/frtb-rrao/docs/PACKAGE_JOURNEY.md), [CVA](../../packages/frtb-cva/docs/PACKAGE_JOURNEY.md), [IMA](../modules/frtb-ima/CLIENT_DELIVERY.md), and [orchestration](../modules/frtb-orchestration/README.md).


## End-to-end user flow

```mermaid
flowchart LR
    engine[Upstream risk engine] --> raw[Arrow or Parquet input tables]
    ref[Reference data and run context] --> raw
    raw --> normalize[Package normalizers]
    normalize --> batch[Canonical package batches]
    batch --> calc[Public calculate_* entrypoints]
    calc --> summary[ComponentCapitalSummary]
    summary --> sa[compose_standardised_approach_capital]
    sa --> suite[calculate_suite_capital or result-store handoff]
    normalize --> reject[Rejected rows and diagnostics]
    calc --> reject
```

The package-owned calculators produce component capital and audit records. The orchestration package composes those outputs; capital packages do not import from sibling capital packages.


## First five minutes

From the repository root:

```bash
uv sync --locked
make demo
make examples-check
make notebooks-check
```

Package demos use public APIs and synthetic inputs. Use them to confirm that your environment can import the package set before wiring real upstream feeds.


## Raw inputs your upstream must emit

| Component | Production grain | Primary public path | Schema and journey |
| --- | --- | --- | --- |
| SBM | One homogeneous sensitivity table per risk class and measure | `normalize_*_arrow_table` -> `build_*_batch_from_arrow` -> `calculate_sbm_capital_from_*_batch` or portfolio dispatcher | [SBM journey](../../packages/frtb-sbm/docs/PACKAGE_JOURNEY.md), [SBM public API](../modules/frtb-sbm/PUBLIC_API.md) |
| DRC | Non-securitisation, securitisation non-CTP, and CTP position tables | `normalize_drc_*_arrow_table` -> `build_drc_*_batch_from_arrow` -> `calculate_drc_capital_from_batch` | [DRC journey](../../packages/frtb-drc/docs/PACKAGE_JOURNEY.md), [DRC public API](../modules/frtb-drc/PUBLIC_API.md) |
| RRAO | Residual-risk position table with classification evidence | `normalize_rrao_arrow_table` -> `build_rrao_batch_from_arrow` -> `calculate_rrao_capital_from_batch` | [RRAO journey](../../packages/frtb-rrao/docs/PACKAGE_JOURNEY.md), [RRAO public API](../modules/frtb-rrao/PUBLIC_API.md) |
| CVA | Counterparty, netting-set, hedge, and SA-CVA sensitivity tables | `normalize_cva_*_arrow_table` -> package batch builders -> BA-CVA or SA-CVA calculators | [CVA journey](../../packages/frtb-cva/docs/PACKAGE_JOURNEY.md), [CVA public API](../modules/frtb-cva/PUBLIC_API.md) |
| IMA | Dense scenario P&L cube plus scenario metadata, RFET observations, and manifests | IMA manifest and public capital assembly entrypoints | [IMA delivery guide](../modules/frtb-ima/CLIENT_DELIVERY.md) |

Generated JSON schema examples live under [`docs/schemas/input_table/`](../schemas/input_table/). Treat those schemas and package `ColumnSpec` tuples as the ETL contract for Tier 1 Arrow or Parquet handoff.


In [ ]:
from datetime import date

from frtb_common import ComponentCapitalSummary, StandardisedComponent
from frtb_orchestration import compose_standardised_approach_capital

AS_OF = date(2026, 5, 30)


def make_summary(component: StandardisedComponent, total: float) -> ComponentCapitalSummary:
    return ComponentCapitalSummary(
        component=component,
        package_name=f"frtb-{component.value.lower()}",
        run_id=f"quickstart-{component.value.lower()}",
        calculation_date=AS_OF,
        base_currency="USD",
        profile_id="US_NPR_2_0",
        total_capital=total,
        profile_hash="synthetic-profile-hash",
        input_hash=f"synthetic-input-{component.value.lower()}",
        line_count=3,
        excluded_line_count=0,
        subtotal_count=1,
        citations=("synthetic-demo-citation",),
        warnings=(),
    )


sbm = make_summary(StandardisedComponent.SBM, 42.0)
drc = make_summary(StandardisedComponent.DRC, 7_875.0)
rrao = make_summary(StandardisedComponent.RRAO, 20_000.0)
sa_result = compose_standardised_approach_capital(
    sbm_summary=sbm,
    drc_summary=drc,
    rrao_summary=rrao,
    run_id="quickstart-sa",
)

print(f"SA total: {sa_result.total_capital:,.2f} {sa_result.base_currency}")
print(f"Jurisdiction family: {sa_result.jurisdiction_family}")
print([item.component.value for item in sa_result.component_subtotals])


## Fail-closed behavior is part of the contract

When profiles, jurisdiction families, required reference data, or unsupported regulatory paths are incomplete, the suite raises explicit errors rather than returning silent zeroes or approximate capital.


In [ ]:
from frtb_orchestration import OrchestrationInputError, standardised_jurisdiction_family

basel_sbm = ComponentCapitalSummary(
    component=StandardisedComponent.SBM,
    package_name="frtb-sbm",
    run_id="quickstart-sbm-basel",
    calculation_date=AS_OF,
    base_currency="USD",
    profile_id="BASEL_MAR21",
    total_capital=42.0,
    profile_hash="synthetic-profile-hash",
    input_hash="synthetic-input-sbm",
    line_count=3,
    excluded_line_count=0,
    subtotal_count=1,
    citations=("synthetic-demo-citation",),
    warnings=(),
)

try:
    compose_standardised_approach_capital(
        sbm_summary=basel_sbm,
        drc_summary=drc,
        rrao_summary=rrao,
    )
except OrchestrationInputError as exc:
    print(f"Rejected mixed jurisdiction families: {exc}")

print(standardised_jurisdiction_family("US_NPR_2_0"))


## Next steps

1. Pick one component and run its `packages/<component>/examples/run_demo.py`.
2. Build or validate an Arrow table matching that component's public `ColumnSpec` and generated schema.
3. Call the package normalizer, batch builder, and capital entrypoint.
4. Convert component results to summaries for Standardised Approach or suite-level orchestration.
5. Keep source row IDs, profile hashes, input hashes, citations, and rejection diagnostics stable so model validation can replay the run.

Useful references: [client integration](../CLIENT_INTEGRATION.md), [visual standards](../visuals.md), [package status dashboard](../quality/PACKAGE_STATUS.md), and [result-store module](../modules/frtb-result-store/README.md).
